# INTERFAZ-IA · Semana 8
## Pt2000 y divisor de voltaje

**Datos de simulación, no resultados experimentales.** Cambie resistencias, ganancia y tensión de alimentación para evaluar el diseño.

In [ ]:
import numpy as np
A=3.9083e-3;B=-5.775e-7;R0=2000.0
VCC=3.3;R_SUP=15000.;R_BAL=2155.87;R_REF_SUP=15000.;VBIAS=0.2

def rtd(t): return R0*(1+A*np.asarray(t)+B*np.asarray(t)**2)
def v_sensor(t): return VCC*rtd(t)/(R_SUP+rtd(t))
def v_reference():return VCC*R_BAL/(R_REF_SUP+R_BAL)
def forward(t,gain):return VBIAS+gain*(v_sensor(t)-v_reference())


### Modelo resistivo y salida de acondicionamiento

\(R(T)=R_0(1+AT+BT^2)\).  Conecte el Pt2000 en la rama inferior y limite la corriente de excitación.

In [ ]:
import matplotlib.pyplot as plt
T=np.linspace(20,100,241)
R=rtd(T)
V=v_sensor(T)
Vout=forward(T,22)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(T,Vout,label='Salida acondicionada (V)')
ax.plot(T,V,label='Nodo Pt2000 (V)')
ax.set(xlabel='Temperatura (°C)',ylabel='Voltaje (V)')
ax.grid(True);ax.legend();plt.show()
print('Corriente máxima sensor (µA):', round(1e6*VCC/(R_SUP+float(rtd(20))),2))
print('Vout min/max (V):',round(Vout.min(),4),round(Vout.max(),4))


### Modelo inverso de calibración

A partir del voltaje calibrado, reconstruya el nodo resistivo, calcule R y resuelva la ecuación cuadrática. El ADC del ESP32 debe calibrarse experimentalmente.

In [ ]:
def inv_pt2000(R):
    c=1-np.asarray(R)/R0
    return (-A+np.sqrt(A*A-4*B*c))/(2*B)
for t in [20,40,60,80,100]:
    print(t, '°C ->',round(float(rtd(t)),3),'Ω ->',round(float(inv_pt2000(rtd(t))),3),'°C')


### Retos
1. Represente la curva en el rango solicitado.
2. Modifique el valor de la resistencia superior y calcule la corriente máxima.
3. Ajuste la ganancia para evitar saturación y deje margen para el ADC.
4. Compare resolución ideal y dispersión observada en los datos reales.
5. Solicite al tutor IA que **revise sus hipótesis**, no que invente lecturas.